# Semantic-ID diagnostics, derived from scratch

Everything we claim about collisions in the 3x1024 SID space, rebuilt from formulas
you can check by hand on tiny synthetic data first, then run on the real 4.78M-item
outputs. Nothing is taken on faith: each section ends with a cross-check against the
published `arm_*_diagnostics.json`.

**Reading order matters.** The metrics form a chain of suspicion, and each section
either convicts or clears one suspect:

| suspect | the metric that judges it | verdict we reached |
|---|---|---|
| not enough address space | uniform-placement floor (Sec 1) | CLEARED (225x oversubscribed) |
| duplicate embeddings in the data | duplicate scan (Sec 1.3) | CLEARED (1.4% of mass) |
| one level not using its codes | marginal entropy / Gini (Sec 2) | CLEARED for real arms (arm A only case) |
| levels restating each other | pairwise MI and total correlation, both vs a shuffled null (Sec 3) | GUILTY -- pairs carry 1.8/0.4/1.1 bits of excess; the triple adds ~0.9 more |
| coupling, specifically | permutation control (Sec 4) | **GUILTY, causally: 28.3% -> 0.35%** |
| tail vs rate | bucket distribution, parent/child split (Sec 5) | different owners (revival vs decoupling) |

The most presentation-worthy moment is Sec 3.2 -- and it is a correction, told honestly.
An earlier draft of this notebook claimed pairwise MI was ~0.05 bits and called the
real dependence XOR-type (invisible pairwise, joint-only). Re-measuring with a
shuffled-pairing null killed that story (the old estimate never had a control and
does not survive remeasurement): the pairs are NOT
innocent (1.8/0.4/1.1 bits of excess), and even the headline "8.3 bits of
redundancy" has a 7.46-bit finite-N floor underneath it. We build the XOR gate first
anyway -- it is the textbook image of dependence a pair can hide from, and the reason
every number below is measured against its null rather than trusted raw.


In [ ]:
%matplotlib inline
import gzip, json, os
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(42)
K, L = 1024, 3                       # codebook size, levels
NOMINAL_SLOTS = K ** L                # 1,073,741,824 addresses

ARMS = {
    "A broken":   "archive/2026-11-14-broken/runs/sid_3x1024_broken/arm_rqvae_index.json",
    "B baseline": "archive/2026-11-07-baseline/runs/sid_3x1024_baseline/arm_rqvae_index.json",
    "C drq":      "archive/2026-11-07-drg/runs/sid_3x1024_baseline/arm_drq_index.json",
    "D warmup":   "archive/2026-11-07-wae/runs/sid_3x1024_baseline/arm_warmup_index.json",
}
DIAG = {
    "B baseline": "archive/2026-11-07-baseline/runs/sid_3x1024_baseline/arm_rqvae_diagnostics.json",
    "C drq":      "archive/2026-11-07-drg/runs/sid_3x1024_baseline/arm_drq_diagnostics.json",
    "D warmup":   "archive/2026-11-07-wae/runs/sid_3x1024_baseline/arm_warmup_diagnostics.json",
}

def load_arm(path):
    opener = gzip.open if os.path.exists(path + ".gz") else open
    real = path + ".gz" if os.path.exists(path + ".gz") else path
    with opener(real, "rt") as f:
        d = json.load(f)
    return np.fromiter((v for row in d.values() for v in row),
                       dtype=np.int16, count=len(d) * 3).reshape(len(d), 3)

SID = {name: load_arm(p) for name, p in ARMS.items()}
for name, c in SID.items():
    print(f"{name:12s} {c.shape}  codes in [0, {c.max()}]")

## 0. The five primitives

Every metric in this notebook is built from these. Define them once, in bits
(log base 2; the JSONs use nats, 1 nat = 1.4427 bits).


In [ ]:
def pack(codes):
    """SID triple -> single int64 key c0*K^2 + c1*K + c2. An 'address'."""
    return (codes[:, 0].astype(np.int64) * K + codes[:, 1]) * K + codes[:, 2]

def counts_of(col):
    return np.bincount(col.astype(np.int64), minlength=K)

def entropy_bits(counts):
    p = counts / counts.sum()
    nz = p > 0
    return float(-(p[nz] * np.log2(p[nz])).sum())

def pair_counts(x, y):
    """Joint histogram of two code columns as a dense (K, K) float matrix."""
    idx = x.astype(np.int64) * K + y.astype(np.int64)
    return np.bincount(idx, minlength=K * K).reshape(K, K).astype(np.float64)

def mi_bits(x, y):
    """I(x;y) = sum_ab P(a,b) log2 [P(a,b)/(P(a)P(b))]."""
    J = pair_counts(x, y)
    J /= J.sum()
    ma, mb = J.sum(1), J.sum(0)
    nz = J > 0
    return float((J[nz] * np.log2(J[nz] / (ma[:, None] * mb[None, :])[nz])).sum())

def gini(counts):
    c = np.sort(counts.astype(np.float64))
    n = c.size
    return float((2 * np.arange(1, n + 1) * c).sum() / (n * c.sum()) - (n + 1) / n)

def tuple_stats(codes):
    keys = pack(codes)
    uniq, cnt = np.unique(keys, return_counts=True)
    N = len(keys)
    return {
        "rho": 1 - len(uniq) / N,                      # collision rate
        "unique": len(uniq),
        "max_bucket": int(cnt.max()),                  # biggest pile on one address
        "items_in_collision": int(cnt[cnt > 1].sum()) / N,
        "counts": cnt,
    }

def tc_bits(codes):
    """Total correlation R = H(c0)+H(c1)+H(c2) - H(c0,c1,c2): duplicated bits in the tuple."""
    h_marg = sum(entropy_bits(counts_of(codes[:, d])) for d in range(L))
    uniq, cnt = np.unique(pack(codes), return_counts=True)
    h_joint = entropy_bits(cnt)
    return h_marg - h_joint

In [ ]:
# Hand-checks -- if these asserts ever fail, the primitives drifted.
uniform = np.full(K, 1000.0)
assert abs(entropy_bits(uniform) - np.log2(K)) < 1e-9          # 10 bits, nothing new
assert abs(gini(uniform)) < 1e-9                                # perfect equality
half = np.concatenate([np.full(K // 2, 2.0), np.zeros(K // 2)])
assert abs(entropy_bits(half) - (np.log2(K) - 1)) < 1e-9        # 9 bits: half the codes dead
assert abs(2.0 ** entropy_bits(half) - K / 2) < 1e-6            # perplexity = effective codes
a_ind = RNG.integers(0, 2, 50_000); b_ind = RNG.integers(0, 2, 50_000)
assert abs(mi_bits(a_ind, b_ind)) < 0.01                        # independent -> MI ~ 0
print('primitives check out')

## 1. Suspect one: not enough address space

The claim to beat: "3x1024 = 1B slots, only 4.8M items, collisions are impossible."
We quantify it. If items were placed **uniformly at random**, the fraction of items
sitting alone in their cell is the classic birthday/coupon quantity:

$$\text{unique fraction}(x) = \frac{1 - e^{-x}}{x}, \qquad x = \frac{N}{K^L}$$

Our $x = 4{,}783{,}154 / 1{,}073{,}741{,}824 = 0.004454$, and for small $x$ the
collision rate $\rho \approx x/2$. Two checks follow: the analytic floor, and a
direct Monte-Carlo simulation of uniform placement on the real $N$.


In [ ]:
N = SID["B baseline"].shape[0]
x = N / NOMINAL_SLOTS
rho_floor = 1 - (-np.expm1(-x)) / x
print(f"load factor x      = {x:.6f}")
print(f"analytic floor     = {100*rho_floor:.3f}%   (~ x/2 = {100*x/2:.3f}%)")

draw = RNG.integers(0, NOMINAL_SLOTS, size=N)          # simulate uniform placement
print(f"simulated floor    = {100*(1 - np.unique(draw).size / N):.3f}%")

print("\nmeasured on the real SIDs:")
TS = {name: tuple_stats(c) for name, c in SID.items()}
for name, t in TS.items():
    print(f"  {name:12s} rho = {100*t['rho']:5.2f}%   max bucket = {t['max_bucket']:4d}"
          f"   unique = {t['unique']:,}")
print(f"\n=> waste factor: {TS['B baseline']['rho']/rho_floor:.0f}x the uniform floor -- capacity is not the problem.")

In [ ]:
def effective_cells(n_items, n_unique):
    """Invert the birthday map: how many UNIFORMLY-used cells would yield this
    many unique tuples? Solves (1 - e^-X)/X = U/N for X, returns M = N/X.
    (This is the 'effective_cells' field in every diagnostics JSON.)"""
    if n_unique >= n_items:
        return float(n_items)
    target = n_unique / n_items
    lo, hi = 1e-9, 1e9
    for _ in range(200):
        mid = np.sqrt(lo * hi)
        if (-np.expm1(-mid)) / mid > target: lo = mid
        else: hi = mid
    return n_items / np.sqrt(lo * hi)

for name, t in TS.items():
    m = effective_cells(N, t["unique"])
    print(f"  {name:12s} nominal 1.07B slots -> actually occupies ~{m:,.0f} cells"
          f"  ({100*m/NOMINAL_SLOTS:.1f}% of the space)")

**Duplicate check (the data's own fault, not the tokenizer's).** Two items with
*identical* 256-d embeddings can never be separated by any deterministic encoder —
that part of the collision mass is irreducible. Measured on the cluster with
`references/scan_duplicate_floor.py` (hash-equal rows over the parquet):

| quantity | value |
|---|---|
| identical-embedding groups | 7,186 |
| items in them | 24,348 (0.51% of corpus) |
| DRQ collision mass (pairs) | 1,230,519 |
| **irreducible** share of that mass | 17,162 pairs = **1.39%** |
| duplicate groups landing in one bucket | 7,186 / 7,186 (all) |

98.6% of collisions are between *distinct* embeddings. Data quality: cleared.
(Also a free determinism check: every duplicate group shares a tuple, exactly as it must.)


## 2. Suspect two (already, sort of): marginals

"Levels don't use their codes." The trap this clears is exactly why our four first-round
improvement arms (norm / jitter / entropy / codebook-lr) showed nothing: they all
target per-level usage, which was *already fine* in every real arm. Watch arm A — the
artificially broken one — to see what marginal pathology actually looks like.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=90)
for name in ["A broken", "B baseline", "C drq"]:
    for d in range(L):
        c = np.sort(counts_of(SID[name][:, d]))[::-1]
        axes[0].plot(np.arange(K), c, label=f"{name} L{d}", alpha=0.7)
axes[0].set_yscale("log"); axes[0].set_title("code usage (sorted, log) -- all 3 levels")
axes[0].set_xlabel("code rank"); axes[0].set_ylabel("items per code")

tab = []
for name in SID:
    row = [name]
    for d in range(L):
        cnt = counts_of(SID[name][:, d])
        row.append(f"L{d}: used={int((cnt>0).sum())}, perp={2**entropy_bits(cnt):.0f}, gini={gini(cnt):.3f}")
    print("  ".join(row))
print("\nperplexity of 1024 -> ~1000 means the marginal is nearly flat.")

## 3. The coupling layer: pairwise MI and total correlation

Two ways to measure "levels restate each other".

**Pairwise** mutual information, from the 1024x1024 co-occurrence table of two levels:

$$I(c_i;c_j)=\sum_{a,b} P(a,b)\log_2\frac{P(a,b)}{P(a)P(b)}$$

**Tuple-level** total correlation — the total "duplicated information" across *all*
levels at once:

$$R = \sum_d H(c_d) - H(c_0, c_1, c_2)$$

$R$ is exactly what an ideal code would drive to zero: three *independent* uniform
levels have $H_{joint} = \sum H_{marg}$. $R$ upper-bounds (and is upper-bounded by
multiples of) the pairwise sum, so if the redundancy were pairwise, $\sum I$ would
catch it.


### 3.1 Two synthetic references — first build one you can *trust*, then build the villain

**Reference 1 (pairwise MI is not blind in general).** A shift-coupling: $c_1 = (c_0 + s) \bmod K$
with random $s$. Uniform marginals, but levels are *strongly* related — pairwise MI
should be big. If our metric reads ~3 bits here, the metric works.

**Reference 2 (the XOR gate).** Three binary variables: $a, b$ independent uniform,
$c = a \oplus b$. Every pair $(a,b), (a,c), (b,c)$ is perfectly independent — MI = 0 —
yet $c$ is *fully determined* by $(a,b)$. Pairwise: innocent. Jointly: guilty.
Keep this as the limiting image of what a pairwise metric can miss. Sec 3.2 will show
our real data is *not* this case -- the pairs are guilty too -- but XOR is exactly why
we never read a dependence number without its null.


In [ ]:
# Reference 1: a coupling with UNIFORM marginals -- Gini can't see it; MI can.
# b = (a+7) mod K: every code equally likely, but b pins a. True MI = log2 K = 10 bits.
# N is large so the estimator floor is small; it is not zero (see next line).
a = RNG.integers(0, K, 3_000_000)
b_shift = (a + 7) % K
b_rand = RNG.integers(0, K, 3_000_000)
print(f"fixed shift:  I(a;b) = {mi_bits(a, b_shift):.3f} bits  (true 10.0; marginals uniform, gini={gini(counts_of(b_shift)):.4f})")
print(f"independent:  I(a;b) = {mi_bits(a, b_rand):.3f} bits  (true 0.0 -- this reading is PURE estimator bias)")
print("  bias floor ~ (K-1)^2 / (2 N ln2) ~ 0.25 bits here; it shrinks as 1/N.")
print("  NOTE: a per-item RANDOM shift (a + s_i) % K scores ~0.3, same as independent --")
print("  a random s_i IS independence by construction. Coupling needs the SAME per-row")
print("  relation. (An earlier draft of this cell used a random shift and concluded the")
print("  metric was blind to coupling. It was blinding itself.)")

# Reference 2: the XOR gate, exactly (no sampling needed)
# H(a)=H(b)=H(c)=1 bit; (a,b,c) takes 4 of 8 states with p=1/4 -> H=2 bits
tc_xor = 1 + 1 + 1 - 2
print(f"\nXOR gate:   I(a;b)=I(a;c)=I(b;c)=0 bits  but  TC = 3 - 2 = {tc_xor} bit exactly")
n = 400_000
xa = RNG.integers(0, 2, n); xb = RNG.integers(0, 2, n); xc = xa ^ xb
joint = (xa * 2 + xb) * 2 + xc
p8 = np.bincount(joint, minlength=8) / n
h_joint = float(-(p8[p8 > 0] * np.log2(p8[p8 > 0])).sum())
h_marg = entropy_bits(np.bincount(xa, minlength=2))            # = 1 bit
print(f"          sampled: I(a;b)={mi_bits(xa,xb):.4f}  I(a;c)={mi_bits(xa,xc):.4f}  I(b;c)={mi_bits(xb,xc):.4f} bits")
print(f"          sampled TC = {3*h_marg - h_joint:.3f} bits  (theory: 1 bit)  -- pairwise says nothing, tuple says everything")

### 3.2 The same two instruments on the real SIDs -- against their nulls

For each arm we measure, then subtract what the metric says when the *pairing is
shuffled* (level-1 and level-2 rows independently permuted; marginals untouched):

- **pairwise excess** $\hat I(c_i;c_j) - \hat I_{\text{shuffled}}$: removes the
  $\sim (K-1)^2/2N\ln 2$ estimator floor (~0.17 bits at $N=4.8$M). Reading raw MI at
  face value is wrong in both directions.
- **TC excess** $R - R_{\text{shuffled}}$: the shuffled $R \approx 7.46$ bits is NOT
  zero-dependence -- it is a **finite-N floor**: $N=4.8$M tuples can carry at most
  $\log_2 N \approx 22.2$ joint bits, while the marginals alone sum to $\approx 29.6$.
  So "8.3 bits of redundancy" overstates the real coupling by ~7.4 bits.

The corrected reading: pairs are *not* innocent (excess ~1.8 / 0.4 / 1.1 bits -- a
residual cascade restates its parent a lot), the triple adds ~0.9 bits beyond any
pair, and the permutation control still convicts the *pairing* as the cause of
collisions, now with the honest effect size (Sec 4: 28.3% -> 0.35%, ~80x).
Consequence for the `mi` arm: its $\gamma\,\hat I(c_d;c_{d+1})$ penalty is NOT
structurally near-blind as we predicted -- it bites on real adjacent redundancy
(~1.8 + 0.4 bits). Whether reducing pairwise redundancy reduces *collisions* is now a
genuine empirical question instead of a foregone conclusion; that makes it a better
experiment, not a worse theory.

**Correction log (visible on purpose).** Two earlier claims, made before this
notebook's nulls existed, are retracted: (a) "pairwise MI ~0.05 bits, so the
dependence is XOR-type, invisible pairwise" -- re-measured with a control, raw
pairwise MI is 1.8/0.4/1.1 bits (A: 2.2/0.8/1.0); (b) "the shuffle control drops
collisions 10x" -- the real drop is ~80x (0.35% floor, not ~3%: the old figure was
an unrun estimate). The XOR demo in Sec 3.1 was also wrong twice over
(a per-item random shift is independence, not coupling, and the plug-in MI at small
N is bias-dominated, showing ~2 bits where the truth is 0). The lesson generalizes:
**any dependence number computed near the support size of its own contingency table
is mostly measuring its estimator.** We keep the mistakes documented because they
are the pedagogical point.


In [ ]:
def shuffled_pairing(codes, seed=0):
    rng_ = np.random.default_rng(seed)
    s = codes.copy(); n = len(codes)
    s[:, 1] = codes[rng_.permutation(n), 1]
    s[:, 2] = codes[rng_.permutation(n), 2]
    return s

print(f"{'arm':12s} {'MI01':>7s} {'MI12':>7s} {'MI02':>7s} {'null':>6s}   {'TC':>6s} {'TCfloor':>7s} {'TCexc':>6s}  {'JSON TC':>7s}")
EXC = {}
for name in ["A broken", "B baseline", "C drq", "D warmup"]:
    c = SID[name]; sh = shuffled_pairing(c)
    i01, i12, i02 = mi_bits(c[:,0], c[:,1]), mi_bits(c[:,1], c[:,2]), mi_bits(c[:,0], c[:,2])
    n01, n12, n02 = mi_bits(sh[:,0], sh[:,1]), mi_bits(sh[:,1], sh[:,2]), mi_bits(sh[:,0], sh[:,2])
    r, rf = tc_bits(c), tc_bits(sh)
    jr = json.load(open(DIAG[name]))["redundancy_bits"] if name in DIAG else float("nan")
    EXC[name] = (i01 - n01, i12 - n12, i02 - n02, r - rf)
    print(f"{name:12s} {i01:7.3f} {i12:7.3f} {i02:7.3f} {(n01+n12+n02)/3:6.3f}   {r:6.3f} {rf:7.3f} {r-rf:6.3f}  {jr:7.3f}")
print("MI columns are RAW; 'null' = mean shuffled-pair MI (the estimator floor); TCexc = TC - TCfloor.")

fig, ax = plt.subplots(figsize=(7.5, 4.3), dpi=90)
names = ["B baseline", "C drq", "D warmup"]
pexc = [sum(EXC[n][:3]) / 3 for n in names]        # mean pairwise excess
exc  = [EXC[n][3] for n in names]                  # TC excess over shuffled floor
w = 0.35; xpos = np.arange(len(names))
ax.bar(xpos - w/2, pexc, w, label="mean pairwise MI excess")
ax.bar(xpos + w/2, exc, w, label="TC excess (beyond any pair)")
for i, (p_, t_) in enumerate(zip(pexc, exc)):
    ax.text(i - w/2, p_, f"{p_:.2f}", ha="center", va="bottom")
    ax.text(i + w/2, t_, f"{t_:.2f}", ha="center", va="bottom")
ax.set_xticks(xpos, names); ax.set_ylabel("bits, over shuffled null"); ax.legend()
ax.set_title("Coupling vs nulls: pairs carry most, triple adds ~0.9")
plt.show()

## 4. The decisive control: shuffle the *pairing*, keep the marginals

If you want one slide to explain the whole project, this is it. Take the trained
baseline SIDs. Independently permute the *rows* of level 1 and level 2 — every code
still appears exactly as often as before (marginals are byte-identical; we assert it),
but the *which-goes-with-which* structure is destroyed.

Predictions, per suspect:
- if collisions were **capacity** -> permuting changes nothing (space is still 1B cells)
- if they were **marginals** -> nothing changes (marginals are preserved exactly)
- if they are **coupling** -> collapse toward the uniform floor of Sec 1

Result: $\rho$: 28.3% -> **0.35%**, max bucket 564 -> 3. (Not the 0.22% analytic
floor because permuted marginals are still unequal -- Sec 1 derived that floor for
uniform marginals.) TC drops 8.31 -> 7.45 bits, and 7.46 is NOT zero: it is the
finite-N floor from Sec 3.2 ($\log_2 N$ joint bits can't hold $\approx 29.6$ marginal
bits, even with no dependence). Same codes, same usage, different pairing, **~78x
fewer collisions**. Coupling is the cause, causally — and the permutation SIDs are
semantically garbage, so this is an attribution device, never a fix.


In [ ]:
c = SID["B baseline"]
sh = c.copy()
sh[:, 1] = c[RNG.permutation(N), 1]
sh[:, 2] = c[RNG.permutation(N), 2]
for d in range(L):                                        # marginals untouched?
    assert np.array_equal(counts_of(sh[:, d]), counts_of(c[:, d]))
t_c, t_s = tuple_stats(c), tuple_stats(sh)
print(f"marginals identical: yes (asserted)")
print(f"collision rate   {100*t_c['rho']:5.2f}%  ->  {100*t_s['rho']:5.2f}%")
print(f"max bucket       {t_c['max_bucket']:5d}   ->  {t_s['max_bucket']}")
print(f"TC (bits)        {tc_bits(c):5.2f}   ->  {tc_bits(sh):5.2f}")

## 5. The tail: who owns which metric

Rate and tail have *different owners* — the reason one headline number was never
going to tell the story. Two EDA plots here are presentation staples:

**(a) Collision-bucket sizes.** Most colliding cells hold a handful of items; a few
hold hundreds. DRQ barely moves the *rate* (28.3 -> 25.7) but decapitates the *tail*
(564 -> 196). Revival moves the rate. Log-log histogram, B vs C.

**(b) Parents and their children (the mechanism).** For a fixed $(c_0, c_1)$ parent,
level 2 splits its $m$ items into at most $K$ children. If splitting tracked *count*,
distinct children would follow $K\,(1 - e^{-m/K})$ (birthday — coupon-collider).
k-means instead spends centroids by residual *variance*, so dense parents get
under-split. Observed-vs-expected scatter is the single best picture of why DRQ's
win is smaller than the paper's: it re-allocates but never escapes the distortion law.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), dpi=90)
for name, col in [("B baseline", "tab:blue"), ("C drq", "tab:orange")]:
    cnt = TS[name]["counts"]
    big = cnt[cnt > 1]
    axes[0].hist(np.log10(big), bins=40, histtype="step", color=col, label=name)
axes[0].set_xlabel("log10 bucket size"); axes[0].set_ylabel("cells")
axes[0].set_title("collision-bucket distribution (cells with >1 item)")
axes[0].legend()

def parent_child(codes):
    parent = codes[:, 0].astype(np.int64) * K + codes[:, 1]
    up, inv = np.unique(parent, return_inverse=True)
    items = np.bincount(inv)                       # items per parent (len = n parents)
    pairs = parent * K + codes[:, 2]
    uniq_pairs = np.unique(pairs)                  # distinct (parent, child) triples
    kids = np.bincount(np.searchsorted(up, uniq_pairs // K), minlength=len(up))
    return up, items, kids

up, items, kids = parent_child(SID["C drq"])
ss = RNG.choice(len(items), size=25_000, replace=False)
m_line = np.linspace(items.min(), items.max(), 100)
axes[1].scatter(items[ss], kids[ss], s=2, alpha=0.15, color="tab:orange")
axes[1].plot(m_line, K * (1 - np.exp(-m_line / K)), color="k", lw=1.5,
             label="uniform split $K(1-e^{-m/K})$")
axes[1].plot(m_line, m_line, color="gray", ls=":", lw=1, label="one child per item")
axes[1].set_xlabel("items in parent $(c_0,c_1)$"); axes[1].set_ylabel("distinct $c_2$ children")
axes[1].set_title("DRQ: dense parents under-split (variance, not count)")
axes[1].legend()
plt.tight_layout(); plt.show()

# quote the band numbers the teammate analysis found
for lo, hi in [(50, 100), (300, 700)]:
    mask = (items >= lo) & (items < hi)
    exp_ = (K * (1 - np.exp(-items[mask] / K))).mean()
    print(f"parents with {lo}-{hi} items: n={mask.sum():5d}  observed children={kids[mask].mean():6.1f}  "
          f"expected={exp_:6.1f}  ratio={kids[mask].mean()/exp_:.2f}")

## 6. The geometry layer (O_pi, K_eff, sigma) — read from JSON, formula here

Centroids aren't archived locally (no .pth), but the definition is short and the
numbers are in `stage[d]` of each diagnostics JSON.

For one level, with usage probabilities $\pi_i$ and codebook geometry:

$$O_\pi = \sum_i \sum_j \pi_i \pi_j \exp\!\left(-\frac{\|c_i - c_j\|^2}{4\sigma^2}\right), \qquad K_{eff} = 1 / O_\pi$$

It is the probability two items drawn by usage end up in overlapping code *balls*.
It splits exactly into `distribution_floor` $= \sum_i \pi_i^2$ (usage skew) and
`geometry_cross_overlap` (centroids sitting on each other). `participation_ratio`
$= 1/\sum_i \pi_i^2$ ignores geometry entirely — it's the ceiling $K_{eff}$ can reach.
$\sigma$ is auto = 0.1 x median nearest-neighbour centroid distance; the `sigma_scan`
sweeps it to check the conclusion doesn't depend on the kernel.

Two reading rules: (i) these are **metric-only** — they never touch training;
(ii) $K_{eff}$ is a *centroid-spacing* statement, while our collisions are a
*count-allocation* statement — that's why cross-overlap is ~1e-9 and yet the tuple
still wastes 95% of its space.


In [ ]:
for name in ["B baseline", "C drq"]:
    d = json.load(open(DIAG[name]))
    print(f"--- {name}")
    for st in d["stage"]:
        print(f"  L{st['stage']}: O_pi={st['overlap_rate']:.2e}  floor={st['distribution_floor']:.2e}  "
              f"geometry={st['geometry_cross_overlap']:.1e}  K_eff={st['effective_codes']:.0f}  "
              f"PR={st['participation_ratio']:.0f}  sigma={st['kernel_sigma']:.4f}")
    print("  sigma scan (mult -> K_eff):")
    for st in d["stage"]:
        line = "  ".join(f"x{e['sigma_mult']:.2f}:{e['effective_codes']:.0f}" for e in st["sigma_scan"])
        print(f"    L{st['stage']}: {line}")
print("\nWatch level 2: PR ~950 (codes spread thin) but K_eff at x4 sigma ~26 -- deep"
      " residuals are geometrically crowded (low-dimensional cone). That's the one"
      " magnitude-flavoured signal we found -- and the reason rank-gauss is worth a flag.")

## 8. References — the names behind each instrument

Everything above is derived from first principles in this notebook, but none of it
was invented here. Citing it is what turns "a metric we made up" into "a diagnostic
with a pedigree", and one item below is a claim of OURS:

| used in | cite | what it is |
|---|---|---|
| Sec 3, TC | Watanabe (1960), *Information Theoretical Analysis of Multivariate Correlation*, IBM J. R&D 4(1):66-82 | origin of total correlation ("multiinformation"); McGill (1954), Psychometrika 19:97-116, is the parallel origin |
| Sec 3, XOR / interaction information | identity $TC = \sum_{pairs} I - II$; $II<0$ = pure higher-order synergy. Bernstein's two-coins-AND-XOR example is the textbook pairwise-independent-yet-jointly-dependent trio | the exact math of the Sec 3.1 demo (note: our real data is NOT the XOR case — II is not ≈ -8, pairs carry real MI) |
| Sec 3, synergy vs redundancy | Rosas, Mediano, Rassouli & Barrett (2020), *Differentiating synergistic and redundant information*, J. R. Soc. Interface 17:20190786 (arXiv:2001.10387) | partial information decomposition; PID literature is young (~2010s) — cite it as framing, not as proof |
| Sec 3.1, estimator bias | Miller (1955), *Note on the Bias of Information Estimates* (+ Paninski 2003, Neural Comput. 15:1191; Victor 2000, Neural Comput. 12:535) | the $(K-1)^2/2N\ln2$ floor behind every raw number in Sec 3-4 |
| Sec 4, permutation control | Fisher (1922), Phil. Trans. R. Soc. A 222:309 (randomization test); Good (1994), *Permutation Tests*, Springer; Anderson (2001, 3rd ed.), *Permutation Tests* | shuffling the pairing while holding the marginals IS the classical permutation null for an $r\times c$ table; the G-test statistic $2N\cdot\hat I \to \chi^2_{(r-1)(c-1)}$ (cf. Sec 3.2's excess numbers) |
| Sec 4, sparse tables | Ráth & Barba (2021), *Sparse contingency tables in complex networks*, Phys. Rev. E 104:044305 | why expected-cell-count rules fail at our sparsity, i.e. why we use a *measured* shuffled null instead of an asymptotic $\chi^2$ p-value |
| Sec 5 / collisions-as-tuple problem | Rajput et al. (2023), TIGER, NeurIPS (arXiv:2305.05065) — collisions fixed by an extra dedup digit, no mechanism analysis; FORGE (arXiv:2509.20904) — post-hoc reassignment; QuaSID (arXiv:2603.00632) — collision-quality tradeoff | the recommender-system prior art: all treat the collision COUNT, none decompose marginal vs joint |
| the phrase "the joint is where the collision is" | — ours | verified zero web/arXiv hits for the exact phrase; the marginal-vs-joint framing is not published in recsys SID work. Claim it |

Two honesty notes: the "Watanabe 1953 conference version" commonly floated is
UNVERIFIED — cite the 1960 IBM journal paper. And the sign convention for
interaction information flips between papers (some define $II$ negated) — state the
definition whenever you cite a sign.


## 7. Cheat sheet — every headline number, where it comes from

| number | formula | computed by |
|---|---|---|
| 0.22% uniform floor | $(1-e^{-x})/x$, $x = N/K^L$ | Sec 1 (analytic + MC) |
| rho = 28.3% / 25.7% | $1 - U/N$ | Sec 1 `tuple_stats`, matches JSON `collision_rate` |
| effective cells ~6.8M | invert birthday map | Sec 1, matches `effective_cells` |
| perplexity ~900-1000 | $2^{H(c_d)}$ | Sec 2, matches `perplexity` |
| Gini 0.17-0.27 | Lorenz formula | Sec 2 |
| TC 8.27-8.34 bits; excess over 7.46-bit floor ~0.9 | $\sum H(c_d) - H(tuple)$; floor = same on shuffled pairing | Sec 3.2, matches `redundancy_bits` |
| pairwise MI excess 1.8 / 0.4 / 1.1 bits (B) | MI $-$ MI on shuffled pairing | Sec 3.2 |
| permuted rho 0.35%, TC floor 7.45 | same metrics after row-shuffle of L1/L2 | Sec 4 |
| max bucket 564 / 196 / 641 / 317 | max cell count | Sec 1 / Sec 5 |
| 121 vs 329 children | observed vs $K(1-e^{-m/K})$ | Sec 5 bands |
| O_pi ~1e-3, K_eff ~930 | overlap kernel on centroid geometry | Sec 6 (from JSON) |
| 1.39% irreducible duplicates | pair mass inside exact-duplicate groups | cluster scan (`references/scan_duplicate_floor.py`) |

**The one-sentence version for a slide:** capacity and marginals check out, the
collisions are pure *pairing* (shuffle the pairing: 28.3% -> 0.35%), measured against
their nulls the pairs carry real redundancy (~1.8/0.4/1.1 bits) with ~0.9 bits
higher-order on top of a 7.46-bit finite-N floor -- and the only mechanism that
re-balances it so far is decoupled assignment buying a 3x lighter tail at a modest
quantization price, the rest of the paper's headline being dead-code revival doing
its job.
